In [1]:
import time
import os
import re
import logging
import pandas as pd
import requests
import random
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager
from urllib.parse import urljoin

In [2]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("sdbs_scraper.log"),
        logging.StreamHandler()
    ]
)

# Define paths
base_dir = r"D:\IITM Intern\irchracterizationcnn--main (3) 2\irchracterizationcnn--main (3)\irchracterizationcnn--main\scripts\Codebase for Github\Scraping Codes\sdbs_data"
other_path = os.path.join(base_dir, 'other')
gif_path = os.path.join(base_dir, 'gif')
ids_path = os.path.join(base_dir, 'sdbs_ids.csv')

# Define URLs
base_url = 'https://sdbs.db.aist.go.jp/CompoundLanding.aspx?sdbsno='
search_url = 'https://sdbs.db.aist.go.jp/sdbs/cgi-bin/direct_frame_top.cgi'
ir_spectra_url_pattern = 'https://sdbs.db.aist.go.jp/sdbs/cgi-bin/landingIR.cgi?sdbsno={}'
ir_image_pattern = 'https://sdbs.db.aist.go.jp/sdbs/cgi-bin/imgdir/ir/{}.gif'

# Define regex patterns for compound information extraction
formula_pattern = re.compile(r'Molecular Formula:[\s]*(.*?)$', re.IGNORECASE)
mw_pattern = re.compile(r'Molecular Weight:[\s]*(.*?)$', re.IGNORECASE)
inchi_pattern = re.compile(r'InChI:[\s]*(InChI=.*?)$', re.IGNORECASE)
inchikey_pattern = re.compile(r'InChI ?Key:[\s]*(.*?)$', re.IGNORECASE)
cas_pattern = re.compile(r'RN:[\s]*(.*?)$', re.IGNORECASE)
name_pattern = re.compile(r'(?:Description:|Compound Name):[\s]*(.*?)$', re.IGNORECASE)

In [3]:
def ensure_directories_exist():
    """Create necessary directories if they don't exist."""
    for directory in [base_dir, other_path, gif_path]:
        if not os.path.exists(directory):
            os.makedirs(directory)
            logging.info(f"Created directory: {directory}")


def load_compound_ids():
    """Load compound IDs from text file."""
    if not os.path.exists(ids_path):
        logging.error(f"IDs file not found: {ids_path}")
        return []
    
    # with open(ids_path, 'r') as f:
    #     return [line.strip() for line in f if line.strip()]

    ids = pd.read_csv(ids_path, header=None)
    return ids[0].astype(str).tolist() 

def setup_driver():
    """Set up and return the Chrome WebDriver with anti-detection measures."""
    chrome_options = Options()
    
    # Add anti-detection measures
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--disable-notifications")
    chrome_options.add_argument("--disable-popup-blocking")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--no-sandbox")
    
    # Set random user agent
    user_agents = [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/15.0 Safari/605.1.15",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:90.0) Gecko/20100101 Firefox/90.0"
    ]
    chrome_options.add_argument(f"--user-agent={random.choice(user_agents)}")
    
    # Experimental options
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    chrome_options.add_experimental_option("prefs", {
        "download.default_directory": gif_path,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": False
    })
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    
    # Execute CDP commands for anti-detection (best effort)
    try:
        driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
            "source": """
            Object.defineProperty(navigator, 'webdriver', {
                get: () => undefined
            });
            """
        })
    except Exception as cdp_e:
        logging.warning(f"CDP anti-detection setup skipped: {cdp_e}")
    
    driver.maximize_window()
    
    return driver


def accept_disclaimer(driver):
    """Handle disclaimer acceptance with multiple strategies."""
    try:
        # Try finding disclaimer button
        disclaimer_selectors = [
            '//input[@value="I agree the disclaimer and use SDBS."]',
            '//input[contains(@value, "agree")]',
            '//button[contains(text(), "agree")]',
            '//a[contains(text(), "agree")]',
            '//input[@type="submit" and contains(@value, "agree")]'
        ]
        
        for selector in disclaimer_selectors:
            try:
                disclaimer_button = WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((By.XPATH, selector))
                )
                logging.info(f"Found disclaimer button with selector: {selector}")
                
                try:
                    disclaimer_button.click()
                    logging.info("Clicked disclaimer button")
                except:
                    logging.info("Direct click failed, trying JavaScript click")
                    driver.execute_script("arguments[0].click();", disclaimer_button)
                
                time.sleep(2)
                return True
            except (TimeoutException, NoSuchElementException) as e:
                logging.debug(f"Disclaimer button not found with selector {selector}: {e}")
        
        # If we reached here, no button was found
        logging.warning("Could not find any disclaimer button with standard selectors")
        return False
        
    except Exception as e:
        logging.error(f"Error handling disclaimer: {e}")
        return False


def navigate_to_compound_page(driver, compound_id, max_retries=3):
    """Navigate to the compound page using the ID with enhanced handling."""
    url = f"{base_url}{compound_id}"

    retries = 0
    while retries < max_retries:
        try:
            driver.get(url)
            logging.info(f"Navigating to URL: {url}")
            time.sleep(3)

            if "disclaimer" in driver.page_source.lower():
                logging.info("Disclaimer found, attempting to accept")
                accept_disclaimer(driver)
                time.sleep(2)

            try:
                driver.execute_script("""
                    if (document.getElementById('CtlMasterHeader_sdbsno')) {
                        document.getElementById('CtlMasterHeader_sdbsno').value = arguments[0];
                    }
                """, compound_id)
            except Exception as form_e:
                logging.debug(f"Error setting form values: {form_e}")

            current_url = driver.current_url
            page_source = driver.page_source

            if compound_id in current_url or f"sdbsno={compound_id}" in current_url:
                logging.info(f"Successfully navigated to compound page for {compound_id}")
                return True

            if "Compound Name:" in page_source or "Molecular Formula:" in page_source:
                logging.info(f"Found compound data on page for {compound_id}")
                return True

            logging.warning(f"Navigation attempt {retries+1}: Not on correct page for compound {compound_id}")
            logging.debug(f"Current URL: {current_url}")

            retries += 1
            time.sleep(2)

        except Exception as e:
            logging.error(f"Error during navigation for compound {compound_id}, attempt {retries+1}: {e}")
            retries += 1
            time.sleep(3)

    logging.error(f"Failed to navigate to compound {compound_id} after {max_retries} attempts")
    return False


def extract_compound_info(driver, compound_id):
    """Extract compound information from compound page."""
    try:
        compound_info = {}
        
        # Extract information using both methods
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        text_content = soup.get_text()
        
        # Method 1: Parse from structured data when available
        for label_elem in soup.find_all(['td', 'div', 'span'], class_=['label', 'name']):
            label_text = label_elem.get_text().strip().lower()
            
            # Look for commonly formatted data
            if 'compound name' in label_text or 'description' in label_text:
                value_elem = label_elem.find_next(['td', 'div', 'span'])
                if value_elem:
                    compound_info['Compound_name'] = value_elem.get_text().strip()
            
            elif 'formula' in label_text:
                value_elem = label_elem.find_next(['td', 'div', 'span'])
                if value_elem:
                    compound_info['Formula'] = value_elem.get_text().strip()
            
            elif 'molecular weight' in label_text:
                value_elem = label_elem.find_next(['td', 'div', 'span'])
                if value_elem:
                    compound_info['Molecular_weight'] = value_elem.get_text().strip()
            
            elif 'cas' in label_text or 'rn' in label_text:
                value_elem = label_elem.find_next(['td', 'div', 'span'])
                if value_elem:
                    compound_info['CAS'] = value_elem.get_text().strip()

        # Method 2: Search text with regex patterns
        for pattern, key in [
            (formula_pattern, 'Formula'),
            (mw_pattern, 'Molecular_weight'),
            (inchikey_pattern, 'InChIKey'),
            (cas_pattern, 'CAS'),
            (name_pattern, 'Compound_name'),
            (inchi_pattern, 'InChI')
        ]:
            if key not in compound_info:  # Only add if not already found
                match = pattern.search(text_content)
                if match and match.group(1).strip():
                    compound_info[key] = match.group(1).strip()
        
        # If we still don't have info, try direct JavaScript extraction
        if not compound_info:
            try:
                # Try to extract using JavaScript if available
                compound_name = driver.execute_script("return document.querySelector('.comp-name')?.textContent || '';")
                formula = driver.execute_script("return document.querySelector('.form-val')?.textContent || '';")
                
                if compound_name:
                    compound_info['Compound_name'] = compound_name.strip()
                if formula:
                    compound_info['Formula'] = formula.strip()
            except Exception as js_e:
                logging.warning(f"JS extraction failed: {js_e}")
        
        # Add the SDBS ID to the information
        compound_info['SDBS_ID'] = compound_id
        
        return compound_info
    
    except Exception as e:
        logging.error(f"Error extracting compound info: {e}")
        return {'SDBS_ID': compound_id}  # At least return the ID


def find_ir_spectra_links(driver, compound_id):
    """Find only IR links (prefer nujol mull, else KBr/disc)."""
    ir_links = []

    def add_link(method, url):
        if not url:
            return
        if url not in [u for _, u in ir_links]:
            ir_links.append((method, url))

    def sanitize_url(raw_url):
        u = (raw_url or '').replace('&amp;', '&').strip()
        while u and u[-1] in ('"', "'", ',', ')', ';', ':', '.'):
            u = u[:-1]
        return u

    def is_target_ir_text(text):
        t = (text or '').lower()
        has_ir = t.startswith('ir') or 'ir :' in t or t.startswith('ir:')
        is_target = ('nujol' in t) or ('nujoll' in t) or ('kbr' in t) or ('disc' in t)
        return has_ir and is_target

    def method_from_text(text):
        t = (text or '').lower()
        if 'nujol' in t or 'nujoll' in t:
            return 'ir_nujol_mull'
        if 'kbr' in t or 'disc' in t:
            return 'ir_kbr_disc'
        return re.sub(r'[^a-zA-Z0-9]+', '_', t).strip('_') or 'ir'

    def extract_irspectralview_url(raw_href, base):
        href = sanitize_url(raw_href)
        if not href:
            return None

        # javascript: ... "https://...IrSpectralView.aspx?..."
        m_abs = re.search(r"https?://[^\s\"'<>)]*IrSpectralView\.aspx[^\s\"'<>)]*", href, re.IGNORECASE)
        if m_abs:
            return sanitize_url(m_abs.group(0))

        # relative IrSpectralView.aspx?...
        m_rel = re.search(r"IrSpectralView\.aspx\?[^\s\"'<>)]*", href, re.IGNORECASE)
        if m_rel:
            return sanitize_url(urljoin(base, m_rel.group(0)))

        if 'IrSpectralView.aspx' in href:
            return sanitize_url(urljoin(base, href))

        return None

    try:
        page_source = driver.page_source
        current_url = driver.current_url
        soup = BeautifulSoup(page_source, 'html.parser')

        # 1) Prefer links in side spectral menu
        spectral_div = soup.find(id='CtlMasterSideMenu_SpectralLink')
        if spectral_div:
            for a_tag in spectral_div.find_all('a'):
                text = ' '.join((a_tag.get_text() or '').split())
                if not is_target_ir_text(text):
                    continue
                ir_url = extract_irspectralview_url(a_tag.get('href', ''), current_url)
                if ir_url:
                    add_link(method_from_text(text), ir_url)

        # 2) Fallback: parse entire page for target IR anchors
        if not ir_links:
            for a_tag in soup.find_all('a'):
                text = ' '.join((a_tag.get_text() or '').split())
                if not is_target_ir_text(text):
                    continue
                ir_url = extract_irspectralview_url(a_tag.get('href', ''), current_url)
                if ir_url:
                    add_link(method_from_text(text), ir_url)

        # 3) Fallback: open IR landing page and parse links
        if not ir_links:
            ir_direct_url = ir_spectra_url_pattern.format(compound_id)
            try:
                driver.get(ir_direct_url)
                time.sleep(2)
                soup_ir = BeautifulSoup(driver.page_source, 'html.parser')
                for a_tag in soup_ir.find_all('a'):
                    text = ' '.join((a_tag.get_text() or '').split())
                    if not is_target_ir_text(text):
                        continue
                    ir_url = extract_irspectralview_url(a_tag.get('href', ''), driver.current_url)
                    if ir_url:
                        add_link(method_from_text(text), ir_url)
            finally:
                try:
                    driver.get(current_url)
                    time.sleep(1)
                except Exception:
                    pass

    except Exception as e:
        logging.error(f"Error finding IR spectra links: {e}")

    # Keep exactly one link: nujol mull first, else KBr/disc.
    if ir_links:
        def _priority(item):
            method = (item[0] or '').lower()
            if 'nujol' in method or 'nujoll' in method:
                return 0
            if 'kbr' in method or 'disc' in method:
                return 1
            return 9

        target = [x for x in ir_links if _priority(x) < 9]
        if target:
            ir_links = [sorted(target, key=_priority)[0]]
            logging.info(f"Selected single IR link for compound {compound_id}: {ir_links[0][0]}")
        else:
            ir_links = []
            logging.info(f"No target IR mode (nujol/disc) found for compound {compound_id}")

    return ir_links


def download_spectrum_image(driver, url, save_path):
    """Open IR page and save only the IR spectrum plot screenshot."""

    def sanitize_url(raw_url):
        u = (raw_url or '').replace('&amp;', '&').strip()
        while u and u[-1] in ('"', "'", ',', ')', ';', ':', '.'):
            u = u[:-1]
        return u

    try:
        url = sanitize_url(url)

        if url.startswith('javascript:'):
            m = re.search(r"https?://[^\s\"'<>)]*IrSpectralView\.aspx[^\s\"'<>)]*", url, re.IGNORECASE)
            if not m:
                logging.warning(f"Skipping non-resolvable javascript URL: {url}")
                return False
            url = sanitize_url(m.group(0))

        logging.info(f"Capturing IR spectrum from URL: {url}")

        if driver.current_url != url:
            driver.get(url)
            time.sleep(2)

        # Pick the largest plot-like visible element from main content area.
        plot_element = driver.execute_script("""
            const elements = Array.from(document.querySelectorAll('img, canvas, svg'));
            const candidates = elements.filter(el => {
                const r = el.getBoundingClientRect();
                if (!r || r.width < 500 || r.height < 220) return false;
                if (r.left < 220) return false; // skip left menu
                if (r.top < 80) return false;
                const style = window.getComputedStyle(el);
                if (style.display === 'none' || style.visibility === 'hidden' || Number(style.opacity) === 0) return false;
                return true;
            });
            candidates.sort((a, b) => {
                const ar = a.getBoundingClientRect();
                const br = b.getBoundingClientRect();
                return (br.width * br.height) - (ar.width * ar.height);
            });
            return candidates.length ? candidates[0] : null;
        """)

        if not plot_element:
            logging.warning(f"No IR plot element found at URL: {url}")
            return False

        plot_element.screenshot(save_path)
        if os.path.exists(save_path) and os.path.getsize(save_path) > 1000:
            logging.info(f"Saved IR plot screenshot to {save_path}")
            return True

        logging.warning(f"Saved file is too small for URL: {url}")
        return False

    except Exception as e:
        logging.error(f"Error capturing spectrum image: {e}")
        return False


def process_compound(driver, compound_id):
    """Process one compound. Returns True only when IR screenshot is saved."""
    logging.info(f"Processing compound ID: {compound_id}")

    other_file_path = os.path.join(other_path, f'{compound_id}_other.txt')
    cid_clean = str(compound_id).strip()
    if cid_clean.endswith('.0'):
        cid_clean = cid_clean[:-2]
    out_path = os.path.join(gif_path, f"{cid_clean}.png")

    # Already done.
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1000:
        logging.info(f"Compound {compound_id} already processed. Skipping...")
        return True

    if not navigate_to_compound_page(driver, compound_id):
        logging.error(f"Failed to navigate to compound page for {compound_id} after multiple attempts")
        return False

    compound_info = extract_compound_info(driver, compound_id)
    if compound_info and len(compound_info) > 1:
        with open(other_file_path, 'w', encoding='utf-8') as file:
            for key, value in compound_info.items():
                file.write(f"{key}: {value}\n")
        logging.info(f"Saved compound information to {other_file_path}")
    else:
        logging.warning(f"No useful information found for compound {compound_id}")

    success = False
    ir_links = find_ir_spectra_links(driver, compound_id)

    if not ir_links:
        logging.warning(f"No IR spectra found for compound {compound_id}")
    else:
        logging.info(f"Found {len(ir_links)} IR spectra for compound {compound_id}")
        method, url = ir_links[0]
        logging.info(f"Downloading spectrum for {compound_id}, method: {method}")
        success = download_spectrum_image(driver, url, out_path)
        if not success:
            logging.warning(f"Failed to download spectrum for {compound_id}, method: {method}")
        elif os.path.getsize(out_path) < 1000:
            logging.warning(f"Downloaded suspiciously small file ({os.path.getsize(out_path)} bytes) for {compound_id}")
            success = False

    try:
        driver.get(search_url)
        time.sleep(2)
    except Exception as nav_e:
        logging.error(f"Error returning to search page: {nav_e}")
        try:
            driver.quit()
        except Exception:
            pass
        return False

    return success


In [4]:
def main():
    """Main function with enhanced error handling and recovery."""
    ensure_directories_exist()
    compound_ids = load_compound_ids()
    
    if not compound_ids:
        logging.error("No compound IDs found. Exiting.")
        return
    
    logging.info(f"Loaded {len(compound_ids)} compound IDs")
    
    # Load already processed IDs
    processed_ids_path = os.path.join(base_dir, "processed_ids.txt")
    processed_ids = set()
    if os.path.exists(processed_ids_path):
        with open(processed_ids_path, 'r') as f:
            processed_ids = set(line.strip() for line in f)
            logging.info(f"Loaded {len(processed_ids)} previously processed IDs")
    
    # Filter out already processed IDs
    compound_ids = [cid for cid in compound_ids if cid not in processed_ids]
    logging.info(f"Remaining compounds to process: {len(compound_ids)}")
    
    if not compound_ids:
        logging.info("All compounds have been processed. Exiting.")
        return
    
    driver = None
    errors = []
    
    try:
        driver = setup_driver()
        
        # Accept initial disclaimer
        driver.get(search_url)
        time.sleep(3)
        accept_disclaimer(driver)
        time.sleep(2)
        
        for i, compound_id in enumerate(compound_ids):
            try:
                logging.info(f"Processing {i+1}/{len(compound_ids)}: {compound_id}")
                success = process_compound(driver, compound_id)
                
                if success:
                    # Add to processed IDs and save
                    with open(processed_ids_path, 'a') as f:
                        f.write(f"{compound_id}\n")
                else:
                    errors.append(compound_id)
                
                # Restart browser periodically to avoid memory issues and detection
                if (i + 1) % 10 == 0:
                    logging.info(f"Restarting browser after processing {i+1} compounds")
                    driver.quit()
                    time.sleep(random.uniform(3, 10))  # Random delay
                    driver = setup_driver()
                    
                    # Accept disclaimer after restart
                    driver.get(search_url)
                    time.sleep(3)
                    accept_disclaimer(driver)
                
                # Random delay between compounds to avoid detection
                time.sleep(random.uniform(2, 5))
                
            except Exception as e:
                logging.error(f"Error processing compound {compound_id}: {e}")
                errors.append(compound_id)
                
                # Restart browser on error
                try:
                    driver.quit()
                except:
                    pass
                
                time.sleep(random.uniform(5, 15))  # Longer delay after error
                driver = setup_driver()
                
                # Accept disclaimer after restart
                driver.get(search_url)
                time.sleep(3)
                accept_disclaimer(driver)
    
    except Exception as e:
        logging.error(f"Fatal error in main process: {e}")
    
    finally:
        if driver:
            try:
                driver.quit()
            except:
                pass
        
        if errors:
            logging.warning(f"Encountered errors with {len(errors)} compounds")
            # Write error IDs to file
            with open(os.path.join(base_dir, "error_ids.txt"), "w") as f:
                for error_id in errors:
                    f.write(f"{error_id}\n")
        
        logging.info("Scraping completed")


if __name__ == "__main__":
    main()


2026-03-06 20:08:14,054 - INFO - Created directory: D:\IITM Intern\irchracterizationcnn--main (3) 2\irchracterizationcnn--main (3)\irchracterizationcnn--main\scripts\Codebase for Github\Scraping Codes\sdbs_data\other
2026-03-06 20:08:14,055 - INFO - Created directory: D:\IITM Intern\irchracterizationcnn--main (3) 2\irchracterizationcnn--main (3)\irchracterizationcnn--main\scripts\Codebase for Github\Scraping Codes\sdbs_data\gif
2026-03-06 20:08:14,068 - INFO - Loaded 31384 compound IDs
2026-03-06 20:08:14,071 - INFO - Remaining compounds to process: 31384
2026-03-06 20:08:14,071 - INFO - ====== WebDriver manager ======
2026-03-06 20:08:15,727 - INFO - Get LATEST chromedriver version for google-chrome
2026-03-06 20:08:15,982 - INFO - Get LATEST chromedriver version for google-chrome
2026-03-06 20:08:16,214 - INFO - Driver [C:\Users\Corei5_8GBRAM_512SSD\.wdm\drivers\chromedriver\win64\145.0.7632.117\chromedriver-win32/chromedriver.exe] found in cache
2026-03-06 20:08:23,158 - INFO - Foun

KeyboardInterrupt: 